# Loading ERA5 Data

> *Pre-requisites*: Code requires most of the packages listed [here](https://github.com/google-research/arco-era5/tree/main/docs/environment.yml):

We will open the Zarr data with XArray after getting GCS permissions. We can test bucket access with fsspec:

In [1]:
import fsspec

fs = fsspec.filesystem('gs')
fs.ls('gs://weatherbench2/datasets/era5/')

['weatherbench2/datasets/era5/',
 'weatherbench2/datasets/era5/1959-2022-1h-240x121_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-1h-360x181_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-128x64_equiangular_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-128x64_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-1440x721.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-240x121_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-512x256_equiangular_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-64x32_equiangular_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-64x32_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-64x33.zarr',
 'weatherbench2/datasets/era5/1959-2022-full_37-1h-0p25deg-chunk-1.zarr-v2',
 'weatherbench2/datasets/era5/1959-2022-full_37-6h-0p25deg-chu

Next, we'll load the 6-hour downsampled data set (including derived variables)

In [2]:
import xarray as xr

reanalysis = xr.open_zarr(
    'gs://weatherbench2/datasets/era5/1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr', 
    chunks={'time': 48},
    consolidated=True,
    decode_timedelta=True    
)

Let's reduce the size by only including data since 1/1/2014 and only our target variables:
- 10m_wind_speed
- 2m_temperature
- high_vegetation_cover
- low_vegetation_cover
- lake_cover
- leaf_area_index_high_vegetation
- leaf_area_index_low_vegetation
- snow_depth
- temperature
- total_precipitation_12hr
- total_precipitation_24hr
- total_precipitation_6hr
- type_of_high_vegetation
- type_of_low_vegetation
- wind_speed
- vorticity

In [3]:
#Select time values since 1/1/2014 and the following variables: 
features = [
    'latitude', 
    'longitude', 
    'time', 
    '10m_wind_speed', 
    '2m_temperature', 
    'high_vegetation_cover', 
    'low_vegetation_cover', 
    'lake_cover', 
    'leaf_area_index_high_vegetation', 
    'leaf_area_index_low_vegetation', 
    'snow_depth', 
    'total_precipitation_12hr', 
    'total_precipitation_24hr', 
    'total_precipitation_6hr', 
    'type_of_high_vegetation', 
    'type_of_low_vegetation'
]

reanalysis = reanalysis.sel(time=slice('2014', '2023'))[features]

Next, we'll restrict our data to county centroids (in the counties_centroids.csv file created by the convert_NWS_shapefile_to_county_centroids notebook).

In the ERA5 coordinate system, latitude values are "normal" but longitude values are expressed as values within [0, 360] with respect to the Greenwich Prime Meridian (i.e., instead of [-180, 180]). Since our county centroid data are all West of the Prime Meridian, we can simply adjust their longitude values with an auxiliary function.

In [4]:
#Use xarray to load the file ../Data/counties_centroids.csv
import pandas as pd
counties_centroids = pd.read_csv('../Data/counties_centroids.csv')

counties_centroids['LON'] = counties_centroids['LON'].astype(float)
counties_centroids['LAT'] = counties_centroids['LAT'].astype(float)

#The function below converts "standard" longitude values to ERA5 longitude values
def lon_to_360(dlon: float) -> float:
  return ((360 + (dlon % 360)) % 360)

counties_centroids['LON'] = counties_centroids['LON'].apply(lon_to_360)

In [5]:
#Create a new DataArray from counties_centroids with the LON and LAT values as longitude and latitude coordinates and FIPS as the data
counties_centroids_da = xr.DataArray(
    counties_centroids['FIPS'].values,
    coords={
        'longitude': ('points', counties_centroids['LON'].values),
        'latitude': ('points', counties_centroids['LAT'].values)
    },
    dims='points'
)

In [6]:
#Restrict the reanalysis data set to coordinates in counties_centroids_da
reanalysis_counties = reanalysis.sel(
    longitude=reanalysis.longitude.isin(counties_centroids_da.longitude),
    latitude=reanalysis.latitude.isin(counties_centroids_da.latitude)
)

We can try saving the zarr file locally as a NetCDF file. The size of the data is roughly 9 GB. However, I've let it run for several hours without it completing.

I've run into roadblocks trying to save locally as a zarr file; I keep getting a "TypeError(f"Expected a BytesBytesCodec. Got {type(data)} instead.")" error. From what I can tell, this appears to be related to zarr 3.

In [ ]:
# Compute the size of the reanalysis_counties dataset
#print(f'size: {reanalysis_counties.nbytes / (1024 ** 4)} TiB')

#Export reanalysis_counties to NetCDF
#reanalysis_counties.to_netcdf('../Data/reanalysis_counties.nc')

size: 0.008082221749646123 TiB


In [8]:
reanalysis_counties

<xarray.Dataset> Size: 9GB
Dimensions:                          (latitude: 94, longitude: 224, time: 13188)
Coordinates:
  * latitude                         (latitude) float32 376B 48.75 ... 24.75
  * longitude                        (longitude) float32 896B 235.8 ... 292.2
  * time                             (time) datetime64[ns] 106kB 2014-01-01 ....
Data variables: (12/13)
    10m_wind_speed                   (time, latitude, longitude) float32 1GB dask.array<chunksize=(44, 94, 224), meta=np.ndarray>
    2m_temperature                   (time, latitude, longitude) float32 1GB dask.array<chunksize=(44, 94, 224), meta=np.ndarray>
    high_vegetation_cover            (latitude, longitude) float32 84kB dask.array<chunksize=(94, 224), meta=np.ndarray>
    low_vegetation_cover             (latitude, longitude) float32 84kB dask.array<chunksize=(94, 224), meta=np.ndarray>
    lake_cover                       (latitude, longitude) float32 84kB dask.array<chunksize=(94, 224), meta=np.ndarray>
    leaf_area_index_high_vegetation  (time, latitude, longitude) float32 1GB dask.array<chunksize=(44, 94, 224), meta=np.ndarray>
    ...                               ...
    snow_depth                       (time, latitude, longitude) float32 1GB dask.array<chunksize=(44, 94, 224), meta=np.ndarray>
    total_precipitation_12hr         (time, latitude, longitude) float32 1GB dask.array<chunksize=(44, 94, 224), meta=np.ndarray>
    total_precipitation_24hr         (time, latitude, longitude) float32 1GB dask.array<chunksize=(44, 94, 224), meta=np.ndarray>
    total_precipitation_6hr          (time, latitude, longitude) float32 1GB dask.array<chunksize=(44, 94, 224), meta=np.ndarray>
    type_of_high_vegetation          (latitude, longitude) float32 84kB dask.array<chunksize=(94, 224), meta=np.ndarray>
    type_of_low_vegetation           (latitude, longitude) float32 84kB dask.array<chunksize=(94, 224), meta=np.ndarray>